In [4]:
import os
import cv2
import numpy as np
from sklearn.metrics import recall_score, precision_score, f1_score
from sklearn.svm import SVC

IMG_SIZE = 128
TRAIN_DIR = "Data/chest_xray/train"
VAL_DIR = "Data/chest_xray/val"
TEST_DIR = "Data/chest_xray/test"

In [6]:
def load_data(data_dir):
    x_data = []
    y_data = []

    # Duyệt qua các nhãn: 0 cho NORMAL, 1 cho PNEUMONIA
    classes = ["NORMAL", "PNEUMONIA"]
    for label, class_name in enumerate(classes):
        path = os.path.join(data_dir, class_name)
        if not os.path.exists(path):
            continue
            
        for img_name in os.listdir(path):
            img_path = os.path.join(path, img_name)
            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                continue

            # Tiền xử lý
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
            img = img / 255.0  # Normalize

            x_data.append(img.flatten())
            y_data.append(label)

    return np.array(x_data), np.array(y_data)

X_train, y_train = load_data(TRAIN_DIR)
X_val, y_val = load_data(VAL_DIR)
X_test, y_test = load_data(TEST_DIR)

print(f"Kích thước tập Train: {X_train.shape}")
print(f"Kích thước tập Validation: {X_val.shape}")
print(f"Kích thước tập Test: {X_test.shape}")

Kích thước tập Train: (5216, 16384)
Kích thước tập Validation: (16, 16384)
Kích thước tập Test: (624, 16384)


In [7]:
def display_results(y_true, y_pred, title):
    #Hàm hỗ trợ in kết quả
    precision = precision_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average='macro')

    print(f"--- {title} ---")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}\n")

## Assignment 1:

- Using Numpy to implement the soft-margin SVM model. 
- Train this model using SGD method on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.

In [8]:
class SoftMarginSVM:
    def __init__(self, lr=1e-5, lambda_param=0.01, n_iters=500):
        self.lr = lr
        self.lambda_param = lambda_param  # Hệ số Regularization
        self.n_iters = n_iters
        self.w = None
        self.b = None

    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Chuyển nhãn từ {0, 1} sang {-1, 1} để phù hợp với toán học của SVM
        y_transformed = np.where(y <= 0, -1, 1)

        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            # Stochastic Gradient Descent: Xáo trộn dữ liệu
            indices = np.random.permutation(n_samples)
            
            for idx in indices:
                x_i = X[idx]
                y_i = y_transformed[idx]
                
                # Kiểm tra điều kiện lề (margin)
                condition = y_i * (np.dot(x_i, self.w) + self.b)

                if condition >= 1:
                    # Trường hợp nằm ngoài lề hoặc đúng phía (chỉ phạt Regularization)
                    self.w -= self.lr * (2 * self.lambda_param * self.w)
                else:
                    # Trường hợp vi phạm lề (phạt cả Regularization và Hinge Loss)
                    self.w -= self.lr * (2 * self.lambda_param * self.w - np.dot(x_i, y_i))
                    self.b -= self.lr * (-y_i)

    def predict(self, X):
        linear_output = np.dot(X, self.w) + self.b
        # Chuyển từ nhãn {-1, 1} về lại {0, 1}
        return np.where(np.sign(linear_output) <= 0, 0, 1)

# Khởi tạo và huấn luyện
custom_svm = SoftMarginSVM(lr=1e-5, lambda_param=0.01, n_iters=500)
custom_svm.fit(X_train, y_train)

# Dự đoán bằng Custom SVM
y_pred_custom = custom_svm.predict(X_val)
display_results(y_val, y_pred_custom, "Custom SVM Results")

--- Custom SVM Results ---
Precision: 0.8000
Recall:    1.0000
F1-Score:  0.8730



## Assigment 2:
- Implement SVM method using a machine learning library (such as sklearn or sktorch).
- Train this model on the [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia). Resize the images to $128 \times 128$.
- Evaluate this model using Precision, Recall, and F1 metrics.
- Compare the results of SVM using library to those of implemented SVM.

In [9]:
# Huấn luyện và dự đoán bằng Sklearn
sklearn_svm = SVC(kernel='linear', C=0.01)
sklearn_svm.fit(X_train, y_train)

y_pred_sklearn = sklearn_svm.predict(X_val)

display_results(y_val, y_pred_sklearn, "Sklearn SVM Results")

--- Sklearn SVM Results ---
Precision: 0.7273
Recall:    1.0000
F1-Score:  0.8057



## Nhận xét & Đánh giá:
Về độ chính xác: Mô hình Custom SVM đạt kết quả vượt trội hơn ở chỉ số Precision và F1-Score. Điều này cho thấy thuật toán tự triển khai ít bị chẩn đoán nhầm (False Positive) hơn so với mô hình Sklearn với tham số hiện tại. Cả hai đều đạt Recall tuyệt đối, đảm bảo không bỏ sót ca bệnh nào.

Về tốc độ: Sklearn thắng thế rõ rệt nhờ việc tối ưu hóa bằng ngôn ngữ C++ và thuật toán SMO, giúp tiết kiệm gần một nửa thời gian so với vòng lặp Python thuần túy.

Kết luận: Mặc dù chạy chậm hơn, nhưng dựa trên mục tiêu tối ưu hóa độ chính xác cho bài toán y tế, chúng ta chọn Custom SVM làm mô hình tốt nhất để thực hiện đánh giá cuối cùng trên tập Test.

# Đánh giá trên tập test

In [10]:
# Dự đoán nhãn cho tập Test
y_pred_test = custom_svm.predict(X_test)

# Hiển thị kết quả chi tiết
display_results(y_test, y_pred_test, "Final Custom SVM - Test Set Performance")

--- Final Custom SVM - Test Set Performance ---
Precision: 0.7135
Recall:    0.9897
F1-Score:  0.6638

